# Sesion 7 — T5 Pandas Avanzado
## Diplomado: Machine Learning en Seguros
### Facultad de Ciencias, UNAM · 29 de abril de 2026  (18:00 - 21:00 h)

---

**Instructor:** Eric   **Sesion:** 7 de 11 del Modulo 1

---

> **Objetivo:** Dominar las operaciones avanzadas de pandas:
> `groupby`, `apply`, `merge` y manejo de datos faltantes (NaN).
> Son las operaciones que usaras en el 80% de tu trabajo actuarial.

> **Nota:** Viernes 1 de mayo NO hay sesion (Dia del Trabajo).
> La siguiente sesion es el **sabado 2 de mayo, 07:00-11:00 h**.

## Contenido

1. [groupby — resumir por grupo](#1-groupby)
2. [.agg() — multiples funciones a la vez](#2-agg)
3. [apply — aplicar cualquier funcion](#3-apply)
4. [merge — unir dos DataFrames](#4-merge)
5. [Datos faltantes: NaN, fillna, dropna](#5-nan)
6. [Ejercicio integrador de cierre](#6-ejercicio)

---
## Datos que usaremos en toda la sesion

Primero construimos la cartera de ejemplo que usaremos en todos los ejemplos:

In [ ]:
import pandas as pd
import numpy as np

# Cartera de polizas de seguros
cartera = pd.DataFrame({
    'poliza':    ['P01','P02','P03','P04','P05','P06','P07','P08'],
    'titular':   ['Ana','Luis','Maria','Carlos','Rosa','Pedro','Elena','Jorge'],
    'edad':      [28,   45,    32,     62,     38,    51,    29,    44],
    'ramo':      ['GMM','Autos','GMM','Vida','Autos','GMM','Autos','Vida'],
    'vehiculo':  [None,'Sedan',None,None,'SUV',None,'Hatchback',None],
    'prima':     [2400, 5800, 1900, 9100, 4100, 3200, 2600, 7800],
    'siniest':   [0,    1,    0,    2,    0,    1,    0,    3],
    'suma_aseg': [300_000,500_000,200_000,1_000_000,400_000,250_000,300_000,800_000],
})

print(f'Cartera: {len(cartera)} polizas, {cartera["prima"].sum():,.0f} en primas')
cartera

---
<a id='1-groupby'></a>
## 1. groupby — Resumir por Grupo

**El problema:** tu jefe pide la prima total por ramo.
Sin groupby tendrias que filtrar cada ramo manualmente.
Con groupby lo haces en una linea.

**Logica:** Split → Apply → Combine
```
Split:   divide el DataFrame en grupos segun la clave
Apply:   aplica una funcion a cada grupo (sum, mean, count...)
Combine: une los resultados en un nuevo DataFrame
```

In [ ]:
# ── El problema sin groupby ──────────────────────────────────────────────────
# Tendrias que hacer esto por cada ramo:
prima_gmm  = cartera[cartera['ramo']=='GMM']['prima'].sum()
prima_autos= cartera[cartera['ramo']=='Autos']['prima'].sum()
prima_vida = cartera[cartera['ramo']=='Vida']['prima'].sum()
print(f'GMM: ${prima_gmm:,} | Autos: ${prima_autos:,} | Vida: ${prima_vida:,}')

# ── Con groupby — una sola linea ─────────────────────────────────────────────
print()
print('Con groupby:')
print(cartera.groupby('ramo')['prima'].sum())

In [ ]:
# ── Funciones de agregacion basicas ──────────────────────────────────────────

# Suma
print('Prima total por ramo:')
print(cartera.groupby('ramo')['prima'].sum())
print()

# Promedio
print('Prima promedio por ramo:')
print(cartera.groupby('ramo')['prima'].mean().round(2))
print()

# Conteo de polizas
print('Numero de polizas por ramo:')
print(cartera.groupby('ramo')['poliza'].count())
print()

# Suma de siniestros
print('Siniestros totales por ramo:')
print(cartera.groupby('ramo')['siniest'].sum())

In [ ]:
# ── Multiples columnas a la vez ──────────────────────────────────────────────
resultado = cartera.groupby('ramo')[['prima','siniest']].sum()
print(resultado)
print()

# ── Agrupar por multiples columnas ───────────────────────────────────────────
# Crear grupo de edad primero
cartera['g_edad'] = pd.cut(
    cartera['edad'],
    bins=[0,30,45,60,100],
    labels=['18-30','31-45','46-60','61+']
)

print('Prima por ramo y grupo de edad:')
print(cartera.groupby(['ramo','g_edad'])['prima'].sum())

In [ ]:
# ── reset_index: convertir el resultado a DataFrame normal ───────────────────
# Por defecto, el resultado de groupby tiene la clave como index
# reset_index() la convierte en columna normal

resumen = (cartera
    .groupby('ramo')['prima']
    .sum()
    .reset_index()
    .rename(columns={'prima':'prima_total'})
)

# Ahora podemos agregar columna de porcentaje
resumen['pct_cartera'] = (resumen['prima_total'] / resumen['prima_total'].sum() * 100).round(1)

print(resumen.to_string())

---
<a id='2-agg'></a>
## 2. .agg() — Multiples Funciones a la Vez

`.agg()` te permite calcular varias estadisticas en una sola llamada,
con nombres de columna personalizados.

In [ ]:
# ── .agg() basico ────────────────────────────────────────────────────────────
resumen = cartera.groupby('ramo').agg(
    polizas    = ('poliza',  'count'),
    prima_total= ('prima',   'sum'),
    prima_prom = ('prima',   'mean'),
    prima_max  = ('prima',   'max'),
    siniest_tot= ('siniest', 'sum'),
).round(2).reset_index()

print(resumen.to_string())

In [ ]:
# ── .agg() con funcion propia ────────────────────────────────────────────────
# Puedes pasar cualquier funcion como agregador

def rango(x):
    return x.max() - x.min()

def coef_variacion(x):
    return x.std() / x.mean() if x.mean() != 0 else 0

analisis = cartera.groupby('ramo')['prima'].agg([
    'count', 'sum', 'mean', 'std', rango, coef_variacion
]).round(2)

print(analisis)

In [ ]:
# ── Reporte ejecutivo completo ───────────────────────────────────────────────
reporte = cartera.groupby('ramo').agg(
    polizas    = ('poliza',  'count'),
    prima_total= ('prima',   'sum'),
    prima_prom = ('prima',   'mean'),
    siniest_tot= ('siniest', 'sum'),
).round(2).reset_index()

reporte['pct_prima']= (reporte['prima_total']/reporte['prima_total'].sum()*100).round(1)
reporte['frec_prom'] = (reporte['siniest_tot']/reporte['polizas']).round(4)

print('=' * 72)
print('  REPORTE POR RAMO — CARTERA DIPLOMADO')
print('=' * 72)
print(reporte.to_string(index=False))
print('=' * 72)
print(f'  TOTAL  {reporte["polizas"].sum():>8}  ${reporte["prima_total"].sum():>10,.2f}')

---
<a id='3-apply'></a>
## 3. apply — Aplicar Cualquier Funcion

`.apply()` recorre cada elemento, columna o fila y aplica una funcion.
Es mas lento que las operaciones vectorizadas, pero te permite usar
**cualquier funcion** — incluyendo las de `mi_modulo.py`.

| Forma | Descripcion | Cuando usar |
|-------|-------------|-------------|
| `serie.apply(func)` | Aplica a cada elemento | Transformar una columna |
| `df.apply(func, axis=1)` | Aplica a cada fila | Logica que necesita varias columnas |

In [ ]:
# ── apply sobre una columna (Serie) ─────────────────────────────────────────
from mi_modulo import clasificar_riesgo, grupo_edad

# Con funcion de mi_modulo
cartera['riesgo'] = cartera['siniest'].apply(clasificar_riesgo)
cartera['grupo']  = cartera['edad'].apply(grupo_edad)

print(cartera[['titular','edad','grupo','siniest','riesgo']].to_string(index=False))

In [ ]:
# ── apply con lambda para logica rapida ──────────────────────────────────────

# Clasificar prima
cartera['nivel_prima'] = cartera['prima'].apply(
    lambda p: 'Alta' if p >= 7000 else 'Media' if p >= 3000 else 'Baja'
)

# Formatear para reporte
cartera['prima_fmt'] = cartera['prima'].apply(lambda p: f'${p:,.0f}')

print(cartera[['titular','prima','nivel_prima']].to_string(index=False))

In [ ]:
# ── apply sobre filas (axis=1) ───────────────────────────────────────────────
# La funcion recibe CADA FILA como una Serie (como un dict)

tarifas_vehiculo = {
    'Sedan':    0.025,
    'SUV':      0.035,
    'Hatchback':0.022,
}

def calcular_prima_autos(fila):
    if fila['ramo'] != 'Autos':
        return None  # solo aplica a autos
    tasa = tarifas_vehiculo.get(fila['vehiculo'], 0.030)
    return fila['suma_aseg'] * tasa * 1.16

cartera['prima_calc_autos'] = cartera.apply(calcular_prima_autos, axis=1)

# Ver solo las polizas de autos
autos = cartera[cartera['ramo']=='Autos'][['titular','vehiculo','suma_aseg','prima_calc_autos']]
print(autos.to_string(index=False))

In [ ]:
# ── Cuando usar apply vs operacion vectorizada ───────────────────────────────

# VECTORIZADA (rapida, preferida cuando es posible):
cartera['prima_con_iva'] = cartera['prima'] * 1.16
cartera['prima_mensual'] = cartera['prima'] / 12

# APPLY (cuando necesitas logica compleja):
# cartera['riesgo'] = cartera['siniest'].apply(clasificar_riesgo)

# Regla practica:
# Si puedes escribirlo como df['col'] * valor → vectorizada
# Si necesitas if/else o varias columnas → apply

print('Prima con IVA (vectorizada):')
print(cartera[['titular','prima','prima_con_iva']].head(3).to_string(index=False))

---
<a id='4-merge'></a>
## 4. merge — Unir Dos DataFrames

En seguros los datos viven en tablas separadas: cartera, tarifas,
siniestros, catalogos de agentes... `merge()` las une por una clave comun.

Es el equivalente del **BUSCARV** de Excel — pero para tablas completas.

In [ ]:
# ── Preparar los DataFrames ──────────────────────────────────────────────────
# Cartera simplificada
polizas = pd.DataFrame({
    'poliza':  ['P01','P02','P03','P04','P05'],
    'titular': ['Ana','Luis','Maria','Carlos','Rosa'],
    'ramo_id': ['R01','R02','R01','R03','R02'],
    'prima':   [2400, 5800, 1900, 9100, 4100],
})

# Catalogo de ramos
ramos = pd.DataFrame({
    'ramo_id':     ['R01',   'R02',    'R03'],
    'nombre_ramo': ['GMM',   'Autos',  'Vida'],
    'cobertura':   [500_000, 300_000, 1_000_000],
})

print('Polizas:')
print(polizas)
print()
print('Ramos:')
print(ramos)

In [ ]:
# ── left merge — el mas usado en actuaria ────────────────────────────────────
# Todas las polizas + la info del ramo donde coincida

resultado = pd.merge(polizas, ramos, on='ramo_id', how='left')
print('Resultado left merge:')
print(resultado.to_string(index=False))
print(f'Filas: polizas={len(polizas)}, resultado={len(resultado)} (se conservan todas)')

In [ ]:
# ── Diferencia entre left, inner y outer ─────────────────────────────────────
# Agregar una poliza sin ramo_id valido para ver la diferencia
polizas_extra = pd.concat([polizas,
    pd.DataFrame([{'poliza':'P06','titular':'Pedro','ramo_id':'R99','prima':3200}])
], ignore_index=True)

# LEFT: conserva todas las polizas, NaN donde no hay ramo
left = pd.merge(polizas_extra, ramos, on='ramo_id', how='left')
print(f'left:  {len(left)} filas  (conserva P06 con NaN en nombre_ramo)')
print(left[['poliza','ramo_id','nombre_ramo']].to_string(index=False))
print()

# INNER: solo polizas con ramo valido
inner = pd.merge(polizas_extra, ramos, on='ramo_id', how='inner')
print(f'inner: {len(inner)} filas  (descarta P06 porque R99 no existe)')

In [ ]:
# ── Claves con nombres distintos en cada tabla ───────────────────────────────
# A veces las tablas usan nombres distintos para la misma clave

polizas2 = pd.DataFrame({
    'id_poliza': ['P01','P02','P03'],
    'titular':   ['Ana','Luis','Maria'],
    'codigo_ramo':['R01','R02','R01'],
})

ramos2 = pd.DataFrame({
    'id_ramo':  ['R01','R02','R03'],
    'ramo_nombre':['GMM','Autos','Vida'],
})

# left_on = columna en el DataFrame izquierdo
# right_on = columna en el DataFrame derecho
res = pd.merge(polizas2, ramos2,
               left_on='codigo_ramo',
               right_on='id_ramo',
               how='left')
print(res[['id_poliza','titular','codigo_ramo','ramo_nombre']].to_string(index=False))

---
<a id='5-nan'></a>
## 5. Datos Faltantes — NaN

**NaN** (Not a Number) representa un valor faltante.
Aparece cuando: lees un CSV con celdas vacias,
haces un merge sin coincidencia, o calculas algo imposible.

**Regla antes de modelar:** siempre revisa y trata los NaN.
Un modelo de ML con NaN en los datos de entrada falla o da resultados incorrectos.

In [ ]:
# ── Crear datos con NaN para practicar ───────────────────────────────────────
df = pd.DataFrame({
    'poliza':   ['P01','P02','P03','P04','P05','P06'],
    'ramo':     ['GMM', None, 'Autos', 'GMM', None, 'Vida'],
    'prima':    [2400, np.nan, 5800, np.nan, 4100, 9100],
    'siniest':  [0, 1, np.nan, 2, 0, np.nan],
    'agente':   ['Ag1','Ag2','Ag1',None,'Ag3','Ag2'],
})

print('DataFrame con valores faltantes:')
print(df)
print()

# Cuantos NaN por columna
print('NaN por columna:')
print(df.isna().sum())
print()

# Porcentaje de NaN
print('Porcentaje de NaN:')
print((df.isna().mean()*100).round(1))

In [ ]:
# ── fillna: rellenar valores faltantes ───────────────────────────────────────

# Rellenar con valor fijo
print('Rellenar siniest con 0:')
print(df['siniest'].fillna(0))
print()

# Rellenar con la mediana (para numericos con outliers)
mediana_prima = df['prima'].median()
print(f'Mediana prima: ${mediana_prima:,.2f}')
df_limpio = df.copy()
df_limpio['prima'] = df_limpio['prima'].fillna(mediana_prima)
df_limpio['siniest'] = df_limpio['siniest'].fillna(0)
df_limpio['ramo'] = df_limpio['ramo'].fillna('Sin ramo')
df_limpio['agente'] = df_limpio['agente'].fillna('Sin asignar')

print('Despues de fillna:')
print(df_limpio.isna().sum())

In [ ]:
# ── dropna: eliminar filas con NaN ───────────────────────────────────────────

print(f'Total filas originales: {len(df)}')

# Eliminar si CUALQUIER columna tiene NaN
sin_nan = df.dropna()
print(f'Despues de dropna(): {len(sin_nan)} filas')

# Eliminar solo si NaN en columnas criticas
solo_prima = df.dropna(subset=['prima'])
print(f'Dropna solo prima: {len(solo_prima)} filas')

# Ver que filas tienen NaN
print()
print('Filas con algun NaN:')
print(df[df.isna().any(axis=1)])

In [ ]:
# ── Duplicados ───────────────────────────────────────────────────────────────
df_con_dup = pd.concat([df, df.iloc[:2]], ignore_index=True)

print(f'Con duplicados: {len(df_con_dup)} filas')
print(f'Duplicados detectados: {df_con_dup.duplicated().sum()}')

# Eliminar duplicados
df_sin_dup = df_con_dup.drop_duplicates()
print(f'Sin duplicados: {len(df_sin_dup)} filas')

# Duplicados por columna especifica
df_sin_dup2 = df_con_dup.drop_duplicates(subset=['poliza'])
print(f'Sin duplicados por poliza: {len(df_sin_dup2)} filas')

---
<a id='6-ejercicio'></a>
## 6. Ejercicio Integrador de Cierre

Combina todo lo visto en la sesion: `groupby`, `apply`, `merge` y manejo de NaN.

In [ ]:
# ── Datos con imperfecciones reales ─────────────────────────────────────────
import pandas as pd
import numpy as np
from mi_modulo import clasificar_riesgo

cartera_ej = pd.DataFrame({
    'poliza':  ['A01','A02','A03','A04','A05','A06'],
    'nombre':  ['Sofia','Miguel','Laura','Roberto','Carmen','Pedro'],
    'ramo_id': ['R01','R02','R01','R03','R02','R01'],
    'prima':   [3200, 5800, np.nan, 9100, 4100, 2600],
    'siniest': [0, 1, 0, 2, 0, np.nan],
    'suma':    [300_000,500_000,250_000,600_000,400_000,280_000],
})

catalogo_ramos = pd.DataFrame({
    'ramo_id':    ['R01',   'R02',   'R03'],
    'nombre_ramo':['GMM',   'Autos', 'Vida'],
    'tasa':       [0.022,   0.035,   0.018],
})
print(f'Datos cargados: {len(cartera_ej)} polizas')

In [ ]:
# ── Tarea 1: Detectar y tratar NaN ───────────────────────────────────────────
# Muestra cuantos NaN hay por columna
# Rellena prima con la mediana y siniest con 0

# Tu codigo aqui:


In [ ]:
# ── Tarea 2: Merge con catalogo de ramos ─────────────────────────────────────
# Haz un left merge de cartera_ej con catalogo_ramos usando 'ramo_id'
# Verifica que el resultado tiene las columnas nombre_ramo y tasa

# Tu codigo aqui:


In [ ]:
# ── Tarea 3: Columnas derivadas con apply ────────────────────────────────────
# Crea la columna 'riesgo' usando clasificar_riesgo() de mi_modulo
# Crea la columna 'prima_calc' = suma * tasa * 1.16 (usando apply axis=1)

# Tu codigo aqui:


In [ ]:
# ── Tarea 4: Reporte con groupby ────────────────────────────────────────────
# Con groupby + .agg() calcula por nombre_ramo:
# - polizas: count
# - prima_total: sum
# - prima_prom: mean
# - siniest_total: sum
# Agrega columna pct_cartera = prima del ramo / total * 100

# Tu codigo aqui:


---
## Resumen de la Sesion 7

| Concepto | Lo que aprendimos |
|---------|------------------|
| **groupby** | Split-Apply-Combine en una linea |
| **.agg()** | Multiples funciones: count, sum, mean, std y propias |
| **apply (Serie)** | Aplicar funcion a cada elemento de una columna |
| **apply (axis=1)** | Aplicar funcion a cada fila — usa varias columnas |
| **merge** | left/inner/outer — unir por clave comun |
| **NaN** | .isna().sum(), .fillna(), .dropna(subset=) |
| **Duplicados** | .duplicated(), .drop_duplicates(subset=) |

**Proxima sesion — Sab 2 de mayo, 07:00-11:00 h (4 hrs):**
T5 continua: leer CSV/Excel/Parquet, `pivot_table`, optimizacion con `dtypes`.

**Tarea:**
```bash
git add sesion7_M1_notebook.ipynb
git commit -m "Sesion 7: groupby apply merge NaN"
git push
```

---
*Diplomado ML en Seguros · Facultad de Ciencias, UNAM · 2026*